In [ ]:
# ================================================================
# USER CONFIGURATION — set these paths for your environment
# ================================================================
repo_path = ''              # e.g. '/content/drive/MyDrive/BachelorsThesis'
dataset_root = ''           # e.g. '/content/drive/MyDrive/BachelorsThesis'

# Root directory for saving anomaly maps — change to your Drive path
# Maps are saved under: {MAPS_SAVE_DIR}/standard/{model_name}/{category}/
MAPS_SAVE_DIR = ''          # e.g. '/content/drive/MyDrive/BachelorsThesis/results/anomaly_maps'
# ================================================================

# Standard Protocol — Real-IAD Benchmark

Evaluates all three DINOv2-based models under the standard multi-view protocol.
All five camera viewpoints are used for both training and evaluation.

Models: AnomalyDINO (memory-based), Dinomaly (reconstruction-based), INP-Former (prototype-based)
Dataset: Real-IAD 512px, 30 categories
Metrics: I-AUROC, S-AUROC, P-AUROC, P-AUPR, AUPRO, Inference Time, Memory Footprint

Note: For a single-category smoke test, see notebooks/01_smoke_test.ipynb

In [ ]:
# Reduce CUDA memory fragmentation
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

from google.colab import drive
import sys

drive.mount('/content/drive')

repo_path = '/content/drive/MyDrive/BachelorsThesis'
dataset_root = '/content/drive/MyDrive/datasets/realiad_512'

# Clone repo if not already present, otherwise pull latest
if not os.path.exists(repo_path):
    !git clone https://github.com/PurpleMono/BachelorsThesis.git {repo_path}
    !git -C {repo_path} submodule update --init
else:
    !git -C {repo_path} pull
    !git -C {repo_path} submodule update --init

# Force INP-Former submodule to correct commit with path fixes
!git -C {repo_path}/models/inp_former fetch origin
!git -C {repo_path}/models/inp_former checkout 6041e2b

sys.path.insert(0, repo_path)

# Install dependencies
!pip install anomalib==2.3.3 ADEval einops timm kornia -q

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Dataset Local Copy

Copies Real-IAD from Google Drive to Colab local SSD for faster I/O during training.
Reading directly from Drive is significantly slower than local storage.
This step is skipped if the data is already present on local storage.

In [ ]:
import shutil

local_dataset_root = '/content/realiad_512'

if not os.path.exists(local_dataset_root):
    print("Copying Real-IAD from Drive to local storage...")
    print("This may take a few minutes but significantly speeds up training I/O...")
    shutil.copytree(dataset_root, local_dataset_root)
    print(f"Dataset ready at: {local_dataset_root}")
else:
    print(f"Dataset already on local storage: {local_dataset_root}")

# Use local path for all training and inference
dataset_root = local_dataset_root
print(f"Active dataset root: {dataset_root}")

In [ ]:
import importlib.util
import pandas as pd
import numpy as np
import gc

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

realiad_utils = load_module("realiad_utils", f"{repo_path}/data/realiad_utils.py")
trainer = load_module("trainer", f"{repo_path}/models/trainer.py")
metrics = load_module("metrics", f"{repo_path}/evaluation/metrics.py")

load_realiad_all = realiad_utils.load_realiad_all
train_dinomaly = trainer.train_dinomaly
train_anomalydino = trainer.train_anomalydino
train_inpformer = trainer.train_inpformer
run_inference = trainer.run_inference
run_inference_inpformer = trainer.run_inference_inpformer
measure_inference_time = trainer.measure_inference_time
measure_memory_footprint = trainer.measure_memory_footprint
compute_i_auroc = metrics.compute_i_auroc
compute_s_auroc = metrics.compute_s_auroc
compute_all_metrics = metrics.compute_all_metrics
REALIAD_CONFIG = metrics.REALIAD_CONFIG

print("All modules loaded")

## Step 1: Load Real-IAD Dataset

Loads all 30 categories from the official JSON-based split.
Training set contains normal images only.
Test set contains both normal and anomalous images.
Labels follow the JSON anomaly_class convention, consistent with all published papers.

In [ ]:
# Load all 30 Real-IAD categories
df = load_realiad_all(data_root=dataset_root)

train_df = df[(df['split'] == 'train') & (df['label'] == 0)]
test_df = df[df['split'] == 'test']

print(f"Categories: {df['category'].nunique()}")
print(f"Train (normal only): {len(train_df)}")
print(f"Test total: {len(test_df)}")
print(f"Test label distribution:\n{test_df['label'].value_counts()}")

# Save split info for reproducibility
os.makedirs(f'{repo_path}/results', exist_ok=True)
os.makedirs(f'{repo_path}/results/weights', exist_ok=True)
df.to_csv(f'{repo_path}/results/dataset_split_standard.csv', index=False)
print("Dataset split saved")

## Step 2: Dinomaly

Reconstruction-based model with frozen DINOv2-Register ViT-Base/14 encoder.
A transformer decoder learns to reconstruct normal encoder features.
Anomaly scores are derived from cosine distance between original and reconstructed features.
Training follows paper defaults: 50,000 iterations, batch size 16, StableAdamW lr=2e-3.

In [ ]:
model_dinomaly = train_dinomaly(
    train_df=train_df,
    n_iterations=50000,
    batch_size=16,
    device='cuda',
    repo_path=repo_path,
    save_path=f'{repo_path}/results/weights/dinomaly_standard.pth'
)

results_dinomaly = run_inference(
    model=model_dinomaly,
    test_df=test_df,
    model_name='Dinomaly',
    device='cuda',
    batch_size=8,
    repo_path=repo_path,
    max_ratio=0.001  # top 0.1% per paper for Real-IAD,
    save_anomaly_maps=True,
    maps_save_dir=f'{MAPS_SAVE_DIR}/standard'
)

results_dinomaly.to_csv(
    f'{repo_path}/results/dinomaly_standard_scores.csv', index=False)
print(f"I-AUROC Dinomaly: {compute_i_auroc(results_dinomaly):.4f}")
print(f"S-AUROC Dinomaly: {compute_s_auroc(results_dinomaly):.4f}")

torch.cuda.empty_cache()
gc.collect()
del model_dinomaly
print("GPU memory cleared")

## Step 3: AnomalyDINO

Training-free nearest-neighbour method with frozen DINOv2-Register ViT-Base/14 encoder.
Patch features from normal training images are stored in a memory bank.
Anomaly scores are computed via nearest-neighbour distance at inference time.
Coreset subsampling is applied to the memory bank for GPU memory compatibility.
The sampling ratio is documented in the methodology section.

In [ ]:
model_anomalydino = train_anomalydino(
    train_df=train_df,
    device='cuda',
    repo_path=repo_path,
    sampling_ratio=0.1,
    save_path=f'{repo_path}/results/weights/anomalydino_standard.pt'
)

results_anomalydino = run_inference(
    model=model_anomalydino,
    test_df=test_df,
    model_name='AnomalyDINO',
    device='cuda',
    batch_size=8,
    repo_path=repo_path,
    save_anomaly_maps=True,
    maps_save_dir=f'{MAPS_SAVE_DIR}/standard'
)

results_anomalydino.to_csv(
    f'{repo_path}/results/anomalydino_standard_scores.csv', index=False)
print(f"I-AUROC AnomalyDINO: {compute_i_auroc(results_anomalydino):.4f}")
print(f"S-AUROC AnomalyDINO: {compute_s_auroc(results_anomalydino):.4f}")

torch.cuda.empty_cache()
gc.collect()
del model_anomalydino
print("GPU memory cleared")

In [ ]:
## Step 4: INP-Former

Prototype-based reconstruction model with frozen DINOv2-Register ViT-Base/14 encoder.
Extends Dinomaly with learnable Image-level Normal Prototype tokens that aggregate
global normal patterns at test time to guide feature reconstruction.
Training follows paper defaults: 200 epochs, batch size 16, StableAdamW lr=1e-3.

In [ ]:
model_inpformer = train_inpformer(
    train_df=train_df,
    dataset_root=dataset_root,
    n_epochs=200,
    batch_size=16,
    device='cuda',
    repo_path=repo_path,
    save_path=f'{repo_path}/results/weights/inpformer_standard.pth'
)

results_inpformer = run_inference_inpformer(
    model=model_inpformer,
    test_df=test_df,
    dataset_root=dataset_root,
    device='cuda',
    batch_size=8,
    repo_path=repo_path,
    save_anomaly_maps=True,
    maps_save_dir=f'{MAPS_SAVE_DIR}/standard'
)

results_inpformer.to_csv(
    f'{repo_path}/results/inpformer_standard_scores.csv', index=False)
print(f"I-AUROC INP-Former: {compute_i_auroc(results_inpformer):.4f}")
print(f"S-AUROC INP-Former: {compute_s_auroc(results_inpformer):.4f}")

torch.cuda.empty_cache()
gc.collect()
del model_inpformer
print("GPU memory cleared")

In [ ]:
## Step 5: Full Evaluation

Computes all metrics for each model and runs Worst-Group Analysis.
Results are saved to the results/ folder for use in the analysis notebook.

In [ ]:
wga_module = load_module("wga", f"{repo_path}/evaluation/wga.py")
wga_by_category = wga_module.wga_by_category
wga_by_viewpoint = wga_module.wga_by_viewpoint
wga_by_defect_type = wga_module.wga_by_defect_type
find_disagreement_groups = wga_module.find_disagreement_groups
print_wga_summary = wga_module.print_wga_summary

# Load saved scores if models were cleared from memory
results_dinomaly = pd.read_csv(
    f'{repo_path}/results/dinomaly_standard_scores.csv')
results_anomalydino = pd.read_csv(
    f'{repo_path}/results/anomalydino_standard_scores.csv')
results_inpformer = pd.read_csv(
    f'{repo_path}/results/inpformer_standard_scores.csv')

# Summary table
summary = pd.DataFrame({
    'Model': ['Dinomaly', 'AnomalyDINO', 'INP-Former'],
    'I-AUROC': [
        compute_i_auroc(results_dinomaly),
        compute_i_auroc(results_anomalydino),
        compute_i_auroc(results_inpformer)
    ],
    'S-AUROC': [
        compute_s_auroc(results_dinomaly),
        compute_s_auroc(results_anomalydino),
        compute_s_auroc(results_inpformer)
    ],
})

print("=" * 50)
print("STANDARD PROTOCOL RESULTS SUMMARY")
print("=" * 50)
print(summary.round(4).to_string(index=False))

summary.to_csv(f'{repo_path}/results/summary_standard.csv', index=False)
print("\nSummary saved to results/summary_standard.csv")

# Worst-Group Analysis
df_dict = {
    'Dinomaly': results_dinomaly,
    'AnomalyDINO': results_anomalydino,
    'INP-Former': results_inpformer
}

print_wga_summary(df_dict)